# 04 — Solver Benchmarks

Mean-CVaR is an LP — there are several solver backends that can
handle it. This notebook compares them on identical instances so
you can see when each one wins.

**Reference doc:** [docs/SOLVER_BACKENDS.md](../../../docs/SOLVER_BACKENDS.md).

## 1. Run the `solver_comparison` benchmark

In [ ]:
from benchmarks.base import load_benchmark_runner

runner = load_benchmark_runner(
    "solver_comparison",
    {
        "n_assets_grid": [25, 50, 100],
        "n_scenarios_grid": [1000, 5000, 10000],
        "backends": ["cpu_cvxpy", "cpu_scipy"],
        "weight_max": 0.30,
    },
)
report = runner.run()
print(f"Benchmark: {report.benchmark_name}")
print(f"Cases: {report.n_cases}  optimal: {report.n_optimal}  failed: {report.n_failed}")

## 2. Tabulate results

In [ ]:
import pandas as pd

rows = [
    {
        "backend": c.backend,
        "solver": c.solver,
        "n_assets": c.n_assets,
        "n_scenarios": c.n_scenarios,
        "solve_ms": c.solve_time_ms,
        "status": c.status,
        "cvar95": c.cvar_95,
    }
    for c in report.cases
]
df = pd.DataFrame(rows)
df_pivot = df.pivot_table(
    index=["n_assets", "n_scenarios"],
    columns="backend",
    values="solve_ms",
)
df_pivot

## 3. Plot solve time vs problem size

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 5))
for backend in df["backend"].dropna().unique():
    sub = df[df["backend"] == backend].sort_values("n_scenarios")
    for n_assets in sub["n_assets"].unique():
        slice_ = sub[sub["n_assets"] == n_assets]
        ax.plot(
            slice_["n_scenarios"], slice_["solve_ms"],
            marker="o", label=f"{backend} (n_assets={n_assets})",
        )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("n_scenarios (log)")
ax.set_ylabel("solve time, ms (log)")
ax.set_title("Mean-CVaR solve time by backend × problem size")
ax.legend(fontsize=8)
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Cross-check objective values

The two backends solve the same LP — their CVaR values should agree
to several decimal places (sign of a healthy LP relaxation).

In [ ]:
df_optimal = df[df["status"].isin(["optimal", "optimal_inaccurate"])]
cvar_pivot = df_optimal.pivot_table(
    index=["n_assets", "n_scenarios"],
    columns="backend",
    values="cvar95",
)
if "cpu_cvxpy" in cvar_pivot.columns and "cpu_scipy" in cvar_pivot.columns:
    diff = (cvar_pivot["cpu_cvxpy"] - cvar_pivot["cpu_scipy"]).abs()
    print(f"Max |Δ CVaR| between backends: {diff.max():.6e}")
cvar_pivot

## 5. Picking a default

- **Small (n ≤ 100, S ≤ 5k):** `cpu_cvxpy` (CLARABEL) wins on setup +
  numerical accuracy.
- **Large (S ≥ 10k):** `cpu_scipy` (HiGHS LP) skips the CVXPY
  canonicalisation overhead and scales better.
- **Cardinality (`max_assets=K`):** would route to `milp_highspy` —
  see the [solver backends doc](../../../docs/SOLVER_BACKENDS.md).